# 🚀 NewsRec Cloud GPU Worker (Free Google Colab T4 GPU)

This notebook runs the **Dense Embedding** (`BAAI/bge-base-en-v1.5`) and **Cross-Encoder Reranker** (`BAAI/bge-reranker-base`) models on a **Free NVIDIA T4 GPU (15GB VRAM)**.

It exposes a secure, high-speed HTTPS tunnel via Cloudflare Tunnel so your local NewsRec platform can offload all heavy neural computation to this GPU in milliseconds.

### Instructions:
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** under Hardware accelerator, then click **Save**.
3. Go to **Runtime** > **Run all** (or press `Ctrl+F9`).
4. Copy the printed `REMOTE_MODEL_URL=https://xxxx.trycloudflare.com` into your local `.env` file!

In [ ]:
# Step 1: Install dependencies & Cloudflare Tunnel binary
!pip install -q fastapi uvicorn sentence-transformers
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Dependencies & Cloudflared installed.")

In [ ]:
# Step 2: Check GPU availability
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"🚀 Active GPU: {gpu_name} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")
else:
    print("⚠️ GPU not detected! Please go to Runtime > Change runtime type and select T4 GPU.")

In [ ]:
# Step 3: Load Neural Models onto GPU
from sentence_transformers import SentenceTransformer, CrossEncoder
device = "cuda" if torch.cuda.is_available() else "cpu"

print("⏳ Loading Embedding Model (BAAI/bge-base-en-v1.5) on GPU...")
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)
embedder.max_seq_length = 256

print("⏳ Loading Cross-Encoder Model (BAAI/bge-reranker-base) on GPU...")
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=160, device=device)
print("✅ Both models loaded into GPU memory ready for high-speed inference!")

In [ ]:
# Step 4: Launch FastAPI Worker & Cloudflare Tunnel
import os
import re
import time
import threading
import subprocess
import numpy as np
from typing import List
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="NewsRec GPU Worker")

class EmbedRequest(BaseModel):
    texts: List[str]

class RerankRequest(BaseModel):
    pairs: List[List[str]]  # [[query, doc], ...]

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "embedding_model": "BAAI/bge-base-en-v1.5",
        "reranker_model": "BAAI/bge-reranker-base"
    }

@app.post("/embed")
def embed(req: EmbedRequest):
    if not req.texts:
        return {"vectors": []}
    vecs = embedder.encode(req.texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    return {"vectors": np.asarray(vecs, dtype=float).tolist()}

@app.post("/rerank")
def rerank(req: RerankRequest):
    if not req.pairs:
        return {"logits": []}
    pair_tuples = [(p[0], p[1]) for p in req.pairs]
    scores = reranker.predict(pair_tuples, batch_size=32, show_progress_bar=False)
    return {"logits": np.asarray(scores, dtype=float).tolist()}

# Start uvicorn in background thread
def start_server():
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

print("Connecting Cloudflare Tunnel...")
public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n" + "=" * 65)
    print("🎉 YOUR CLOUD GPU WORKER IS ONLINE!")
    print(f"REMOTE URL: {public_url}")
    print("=" * 65)
    print(f"\n👉 Copy this line into your local .env file:\n")
    print(f"REMOTE_MODEL_URL={public_url}\n")
    print("Keep this Colab tab running while using the news recommender.")
else:
    print("Could not start tunnel automatically. Please re-run this cell.")